# Phase 10 — Monitoring & Détection de Drift

La fiabilité du pipeline SON dépend de la qualité des prédictions ML. Le *Data Drift* (changement dans les données d'entrée) ou le *Concept Drift* (évolution du comportement des utilisateurs) dégradent la précision avec le temps.

## 👁️ Stratégie de détection
Pour monitorer le modèle sans labels temps réel (souvent indisponibles instantanément), nous utilisons :
- **Suivi du résidu STL** : Le résidu de la décomposition STL (Phase 2) est l'indicateur d'anomalie idéal (le signal stationnaire attendu).
- **Rolling MAE (Mean Absolute Error)** : Suivi de l'erreur sur une fenêtre glissante de 48 slots (24h).
- **Test de Page-Hinkley** : Algorithme classique de détection de changement de moyenne sur le signal d'erreur, permettant de détecter une dégradation progressive.

In [ ]:
import numpy as np
from collections import deque

class DriftMonitor:
    def __init__(self, window=48, threshold_sigma=2.0, consecutive=5):
        self.window = window
        self.threshold = threshold_sigma
        self.consecutive = consecutive
        self.errors = deque(maxlen=window)
        self.anomaly_count = 0
        self.alerts = []

    def update(self, slot, predicted, actual, stl_residual):
        # Calcul de l'erreur absolue
        err = abs(predicted - actual)
        self.errors.append(err)
        
        # Détection d'anomalie via résidu STL (si |résidu| > k * std(erreurs))
        if len(self.errors) < 10: return False
        is_anomaly = abs(stl_residual) > self.threshold * np.std(self.errors)
        
        if is_anomaly:
            self.anomaly_count += 1
        else:
            self.anomaly_count = 0
            
        # Alerte si dérive persistante
        if self.anomaly_count >= self.consecutive:
            alert = {
                'slot': slot, 
                'rolling_mae': np.mean(self.errors),
                'stl_residual': stl_residual, 
                'type': 'DRIFT_DETECTED'
            }
            self.alerts.append(alert)
            print(f'[DRIFT ALERT] Dérive détectée au slot {slot} ! Rolling MAE: {np.mean(self.errors):.2f}')
            self.anomaly_count = 0
            return True
        return False

# Test rapide
monitor = DriftMonitor()
print("Moniteur de drift initialisé.")